<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Probar Varias Recetas sin Hacer Trampa: Validación Cruzada 🔄
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 03
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/03%20-%20Metricas%20y%20Ajuste%20de%20Hiperparametros/Para%20Dummies/02_Validacion_Cruzada_GridSearch_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 02 (Validación Cruzada y GridSearch)](../02_Validacion_Cruzada_y_GridSearch.ipynb). Todo modelo tiene «perillas» que alguien debe girar antes de entrenar (cuántos vecinos mirar, qué tan profundo preguntar...). ¿Cómo encontrar **la mejor combinación sin hacer trampa**? Con una idea muy conocida por cualquier estudiante: **los simulacros antes del examen final**.

Al terminar podrás explicar:
1. Por qué **no** se ajustan las perillas mirando el examen final.
2. Cómo funciona la **validación cruzada** (dividir en grupos y rotar).
3. Qué es la **fuga de datos** («ver las respuestas antes») y cómo evitarla.
4. Cómo **GridSearch** prueba todas las combinaciones, como un cocinero que prueba recetas.
5. Por qué probar demasiadas combinaciones **se vuelve carísimo**.

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../02_Validacion_Cruzada_y_GridSearch.ipynb). Otras miradas sencillas: [Selección de modelos y validación cruzada (Dummies) en Data Science Programming](../../../Data%20Science%20programming/07%20-%20Regression/Para%20Dummies/04_Seleccion_Modelos_Validacion_Cruzada_y_KNN_Dummies.ipynb) y [k-NN y selección de modelos (Dummies)](../../../Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/04_KNN_Clasificacion_y_Seleccion_Modelos_Dummies.ipynb).

---
## 1. Simulacros y examen final: no se vale mirar el examen 📝

Para preparar un examen haces **simulacros**. Si por cada simulacro mirases el **examen final real** para ajustar cómo estudias, tu nota final **ya no demostraría lo que sabes**: habrías estudiado *para ese examen en particular*.

Con los modelos pasa igual. Hay dos tipos de «números»:

- **Lo que el modelo aprende solo** al estudiar los datos (por ejemplo, los pesos que asigna a cada variable).
- **Las perillas que giras tú antes de empezar** (por ejemplo, «cuántos vecinos consulto»). Se llaman **hiperparámetros**.

Para elegir las perillas necesitas **simulacros**, no el examen final. El **examen final** (los datos de prueba) se abre **una sola vez**, al terminar, para dar la nota definitiva.

Veamos qué pasa cuando alguien **hace trampa**: datos donde **no hay nada que aprender** (las respuestas son al azar), pero se prueban 300 «modelos» y se queda con el mejor *mirando el examen*.

In [ ]:
# Hacer trampa: elegir el mejor de 300 modelos mirando el examen, con datos donde NO hay nada que aprender
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(42)
X = rng.normal(size=(600, 30))
y = rng.integers(0, 2, 600)                                              # respuestas al AZAR: ningun modelo puede acertar mas del 50 % en promedio

X_estudio, y_estudio = X[:200], y[:200]
X_examen_1, y_examen_1 = X[200:400], y[200:400]                          # examen que el tramposo MIRA para elegir
X_examen_2, y_examen_2 = X[400:], y[400:]                                # examen NUEVO que nadie ha visto

notas_examen_1, notas_examen_2 = [], []
for _ in range(300):                                                     # 300 "modelos": cada uno usa 5 columnas al azar
    columnas = rng.choice(30, size=5, replace=False)
    m = LogisticRegression().fit(X_estudio[:, columnas], y_estudio)
    notas_examen_1.append(m.score(X_examen_1[:, columnas], y_examen_1))
    notas_examen_2.append(m.score(X_examen_2[:, columnas], y_examen_2))

mejor = int(np.argmax(notas_examen_1))
print(f"Nota del 'mejor' modelo en el examen que el tramposo miró : {notas_examen_1[mejor]:.1%}")
print(f"Nota de ESE mismo modelo en un examen nuevo               : {notas_examen_2[mejor]:.1%}")
print("\nCon respuestas al azar, lo honesto es ~50 %. El tramposo 'descubrió' un talento que no existe, solo porque eligió mirando el examen.")

---
## 2. Validación cruzada: simulacros rotativos 🔄

Para no gastar el examen final, haces **varios simulacros** y ocupas **todos tus datos** de forma inteligente:

1. Divide tus datos de estudio en **5 grupos** iguales.
2. **Simulacro 1:** estudia con los grupos 2, 3, 4 y 5; **examínate con el grupo 1**.
3. **Simulacro 2:** estudia con 1, 3, 4 y 5; examínate con el grupo 2.
4. ...y así hasta que **cada grupo haya sido el examen una vez**.
5. Tu nota es el **promedio** de los 5 simulacros.

Eso es la **validación cruzada** (*k-fold*, con k = 5). Aprovecha todos los datos y la nota **depende menos de la suerte** de un solo reparto. Veamos cómo rota quién es «el grupo de examen» con 10 datos y 5 grupos.

Y ahora una pregunta práctica: **¿qué perilla elijo?** Un modelo sencillo llamado **vecinos más cercanos** decide mirando a sus **k vecinos más parecidos**. Si preguntas a **1 solo vecino**, te dejas llevar por casos raros (es como memorizar); si preguntas a **demasiados**, ya no distingues nada. Si eliges k por la nota con los **mismos datos con los que estudió**, siempre ganará k = 1 (¡memoriza y saca 100 %!). Hay que elegir con los **simulacros**. Primero vemos cómo rotan los grupos (10 datos, 5 grupos) y luego elegimos k con datos de **tumores**.

In [ ]:
# Como rotan los grupos en una validacion cruzada de 5 grupos con 10 datos
from sklearn.model_selection import KFold

datos = np.arange(10)                                                    # 10 "datos" numerados del 0 al 9
for numero, (estudio, examen) in enumerate(KFold(n_splits=5).split(datos), start=1):
    print(f"Simulacro {numero}: estudia con {datos[estudio]} | se examina con {datos[examen]}")
print("\nCada dato se usa exactamente UNA vez para examinar y las otras veces para estudiar.")

# Elegir k: nota con los datos de estudio (engaña) vs. validacion cruzada (honesta), con datos de tumores
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

datos_tumores = load_breast_cancer()
valores_k = list(range(1, 32, 2))
nota_estudio, nota_simulacros = [], []
for k in valores_k:
    modelo = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    nota_estudio.append(modelo.fit(datos_tumores.data, datos_tumores.target).score(datos_tumores.data, datos_tumores.target))      # examinado con lo mismo que estudio
    nota_simulacros.append(cross_val_score(modelo, datos_tumores.data, datos_tumores.target, cv=5).mean())          # promedio de 5 simulacros

mejor_k = valores_k[int(np.argmax(nota_simulacros))]
plt.figure(figsize=(9, 4.8))
plt.plot(valores_k, nota_estudio, "o-", color="#dc2626", label="Nota con los mismos datos que estudió (engaña)")
plt.plot(valores_k, nota_simulacros, "s-", color="#16a34a", label="Nota promedio de los 5 simulacros (honesta)")
plt.axvline(mejor_k, color="gray", linestyle=":", label=f"Mejor k según simulacros: {mejor_k}")
plt.xlabel("Número de vecinos consultados (k)"); plt.ylabel("Aciertos")
plt.title("Elegir con simulacros, no con lo que ya vio el modelo", fontweight="bold")
plt.legend(fontsize=9)
plt.show()
print(f"Con k = 1 la nota 'de estudio' es {nota_estudio[0]:.0%} (memoriza), pero en simulacros solo {nota_simulacros[0]:.1%}.")
print(f"Los simulacros recomiendan k = {mejor_k}, con {max(nota_simulacros):.1%} de aciertos honestos.")

---
##### 🛠️ Práctica 1: Rota tú los simulacros

Ya viste cómo rotan los grupos con 10 datos y 5 simulacros. Ahora hazlo con **12 datos y 4 simulacros** (cada grupo tendrá 3 datos):

1. Crea `datos12 = np.arange(12)`.
2. Recorre `KFold(n_splits=4).split(datos12)` e imprime, en cada simulacro, con quién estudia y con quién se examina (como en el ejemplo).
3. Lleva la cuenta en un arreglo `veces_examen` (empieza en ceros): en cada simulacro suma 1 en las posiciones de `examen`.
4. Comprueba con `assert` que **cada dato se examinó exactamente una vez** y que en cada simulacro estudian 9 datos y se examinan 3.

In [ ]:
# Escribe tu código aquí

# datos12 = np.arange(12)
# veces_examen = np.zeros(12, dtype=int)
#
# for numero, (estudio, examen) in enumerate(KFold(n_splits=...).split(datos12), start=1):
#     print(f"Simulacro {numero}: estudia con {datos12[estudio]} | se examina con {datos12[examen]}")
#     veces_examen[examen] += 1
#     assert len(estudio) == ... and len(examen) == ...
#
# assert (veces_examen == ...).all()


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Doce datos numerados del 0 al 11
datos12 = np.arange(12)
veces_examen = np.zeros(12, dtype=int)                                   # cuantas veces se examino cada dato

# 2-3) Cuatro simulacros: en cada uno rota quien es el grupo de examen
for numero, (estudio, examen) in enumerate(KFold(n_splits=4).split(datos12), start=1):
    print(f"Simulacro {numero}: estudia con {datos12[estudio]} | se examina con {datos12[examen]}")
    veces_examen[examen] += 1
    assert len(estudio) == 9 and len(examen) == 3                        # 9 estudian, 3 se examinan

# 4) Cada dato se examina exactamente una vez
assert (veces_examen == 1).all()
print("\nCada dato se examinó exactamente una vez y estudió las otras tres veces.")
```
</details>

---


---
## 3. La fuga de datos: ver las respuestas antes 🚰

Una trampa muy común y **difícil de notar** se llama **fuga de datos** (*data leakage*). Ocurre cuando, **antes** de hacer los simulacros, usas **toda** la información (incluidas las respuestas del grupo que luego será «examen») para **preparar** el material de estudio. Es como si, antes de un simulacro, alguien te dijera *«estas son las 20 preguntas que más pesan en el examen»* mirando las respuestas del examen: así cualquiera saca buena nota.

La **solución**: todo lo que **aprende de los datos** (escalar, elegir las variables más útiles, etc.) debe hacerse **dentro** de cada simulacro, solo con los grupos de estudio. En `scikit-learn` esto se hace con un **`Pipeline`** (una «línea de ensamblaje» que cuida el orden). Lo probamos con **datos de puro azar**: cualquier acierto por encima del 50 % es un espejismo.

In [ ]:
# Fuga de datos: elegir variables ANTES de los simulacros (trampa) vs. DENTRO de cada simulacro (correcto), con datos de puro azar
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

rng = np.random.default_rng(0)
X = rng.normal(size=(60, 2000))                                          # 60 personas, 2000 columnas de puro azar
y = rng.integers(0, 2, 60)                                               # respuestas al azar: no hay nada que aprender

# TRAMPA: elegir las 20 columnas "mas utiles" mirando TODAS las respuestas, y luego hacer los simulacros
X_elegidas = SelectKBest(f_classif, k=20).fit_transform(X, y)
nota_trampa = cross_val_score(LogisticRegression(), X_elegidas, y, cv=5).mean()

# CORRECTO: la eleccion de columnas ocurre dentro de cada simulacro, solo con los grupos de estudio
linea = Pipeline([("elegir", SelectKBest(f_classif, k=20)), ("modelo", LogisticRegression())])
nota_correcta = cross_val_score(linea, X, y, cv=5).mean()

print(f"Con trampa (fuga de datos): {nota_trampa:.0%} de aciertos  <- ¡un espejismo!")
print(f"Haciéndolo bien (Pipeline) : {nota_correcta:.0%} de aciertos  <- lo honesto con datos de azar es ~50 %")

---
## 4. GridSearch: el cocinero que prueba todas las recetas 🍳

Tienes un modelo con **dos perillas** (por ejemplo, «C» y «gamma» en un modelo llamado SVM). ¿Qué hace **GridSearch**? Es como un cocinero que quiere la mejor receta de arepas y decide probar **todas las combinaciones** de, digamos, 4 cantidades de sal y 5 tiempos de cocción: **4 × 5 = 20 recetas**. Cada receta la prueba con los **simulacros** (validación cruzada), anota la nota promedio y se queda con la **mejor**. Al final reentrena esa receta con todos los datos de estudio y **solo entonces** abre el examen final, una sola vez.

**Cuidado: ¡demasiadas recetas!** Con **dos** perillas y 5 opciones cada una son 25 recetas, pero cada perilla nueva **multiplica** el trabajo por 5 (con 5 simulacros por receta): 5 perillas son 3 125 recetas (15 625 entrenamientos) y 8 perillas, casi **2 millones** de entrenamientos. Por eso existen atajos (probar combinaciones **al azar**, o ir **aprendiendo** de las ya probadas) que verás en el **Módulo 07**.

In [ ]:
# GridSearch: probar 4 x 5 = 20 combinaciones de dos perillas con simulacros (validacion cruzada)
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

X_estudio, X_final, y_estudio, y_final = train_test_split(datos_tumores.data, datos_tumores.target, test_size=0.25, stratify=datos_tumores.target, random_state=42)
receta = Pipeline([("escalar", StandardScaler()), ("modelo", SVC())])
perillas = {"modelo__C": [0.1, 1, 10, 100], "modelo__gamma": [0.0001, 0.001, 0.01, 0.1, 1]}

busqueda = GridSearchCV(receta, perillas, cv=5).fit(X_estudio, y_estudio)
tabla = pd.DataFrame(busqueda.cv_results_)[["param_modelo__C", "param_modelo__gamma", "mean_test_score"]]
tabla.columns = ["C", "gamma", "nota promedio en simulacros"]

print(f"Recetas probadas: {len(tabla)} (cada una con 5 simulacros = {len(tabla) * 5} entrenamientos)\n")
print("Las 5 mejores combinaciones:")
print(tabla.sort_values("nota promedio en simulacros", ascending=False).head(5).round(3).to_string(index=False))
print(f"\nMejor receta: {busqueda.best_params_}")
print(f"Nota en los simulacros: {busqueda.best_score_:.1%}")
print(f"Nota en el examen final (se abre UNA sola vez): {busqueda.score(X_final, y_final):.1%}")

# Cuantas recetas y entrenamientos cuesta GridSearch segun el numero de perillas (5 opciones cada una, 5 simulacros)
for perillas_n in [2, 3, 5, 8]:
    recetas = 5 ** perillas_n
    print(f"{perillas_n} perillas -> {recetas:>8,} recetas -> {recetas * 5:>9,} entrenamientos")
print("\nCada perilla extra multiplica el costo por 5: crece muy, muy rápido.")

---
##### 🛠️ Práctica 2: Tu receta con vecinos: GridSearch de k y del peso

Ahora eres tú el cocinero. Con los datos de estudio y el examen final de la Sección 4 (`X_estudio`, `y_estudio`, `X_final`, `y_final`) prueba un modelo de **vecinos más cercanos** con dos perillas:

1. Arma `receta_knn`: un `Pipeline` con dos pasos, `("escalar", StandardScaler())` y `("modelo", KNeighborsClassifier())`.
2. Define las perillas `perillas_knn`: `"modelo__n_neighbors"` con `[1, 5, 11, 21]` y `"modelo__weights"` con `["uniform", "distance"]` (¡4 × 2 = 8 recetas!).
3. Ejecuta `busqueda_knn = GridSearchCV(receta_knn, perillas_knn, cv=5).fit(X_estudio, y_estudio)`.
4. Imprime la mejor receta, su nota en los simulacros y la nota en el examen final (**una sola vez**, con `busqueda_knn.score(X_final, y_final)`).
5. Comprueba con `assert` que se probaron 8 recetas y que la nota de los simulacros de la mejor receta es de al menos $0.90$.

In [ ]:
# Escribe tu código aquí

# receta_knn = Pipeline([("escalar", ...), ("modelo", ...)])
# perillas_knn = {"modelo__n_neighbors": [...], "modelo__weights": [...]}
# busqueda_knn = GridSearchCV(receta_knn, perillas_knn, cv=5).fit(X_estudio, y_estudio)
# print(f"Mejor receta: {busqueda_knn.best_params_}")
# print(f"Nota en simulacros: {busqueda_knn.best_score_:.1%}")
# print(f"Nota en el examen final: {busqueda_knn.score(X_final, y_final):.1%}")
# assert len(busqueda_knn.cv_results_["params"]) == ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) La receta: escalar y luego preguntar a los vecinos
receta_knn = Pipeline([("escalar", StandardScaler()), ("modelo", KNeighborsClassifier())])

# 2) Las dos perillas: 4 valores de k x 2 formas de pesar a los vecinos = 8 recetas
perillas_knn = {"modelo__n_neighbors": [1, 5, 11, 21], "modelo__weights": ["uniform", "distance"]}

# 3) GridSearch: prueba las 8 recetas con 5 simulacros cada una (solo con los datos de estudio)
busqueda_knn = GridSearchCV(receta_knn, perillas_knn, cv=5).fit(X_estudio, y_estudio)

# 4) Resultados; el examen final se abre UNA sola vez
print(f"Mejor receta: {busqueda_knn.best_params_}")
print(f"Nota en los simulacros: {busqueda_knn.best_score_:.1%}")
print(f"Nota en el examen final: {busqueda_knn.score(X_final, y_final):.1%}")

# 5) Comprobaciones
assert len(busqueda_knn.cv_results_["params"]) == 8
assert busqueda_knn.best_score_ >= 0.90
```
</details>

---


---
## Resumen en una frase 🎯

> **Para elegir bien las perillas de un modelo se usan simulacros (validación cruzada) en lugar del examen final, se evita la fuga de datos metiendo todos los pasos en una línea de ensamblaje (Pipeline), y GridSearch prueba todas las combinaciones, con cuidado de que no sean demasiadas.**

### Lo que aprendiste hoy:
- ✅ **Nunca** ajustes las perillas mirando el examen final: te dará una nota inflada.
- ✅ La **validación cruzada** hace simulacros rotativos y aprovecha todos los datos.
- ✅ La **fuga de datos** («ver las respuestas antes») fabrica notas falsas; el **Pipeline** la evita.
- ✅ **GridSearch** prueba todas las combinaciones y reentrena la mejor; el examen final se abre **una vez**.
- ✅ Cada perilla nueva **multiplica** el costo de la búsqueda.

> 🎓 **Siguiente paso:** el [Taller Práctico del módulo](../../homeworks/03_Metricas_y_Ajuste_Hands_On.ipynb). Versión con más detalle de este tema: [cuaderno estándar](../02_Validacion_Cruzada_y_GridSearch.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué sería trampa ajustar las perillas de tu modelo mirando el examen final, aunque «solo fuera un par de veces»?
2. Piensa en cómo te preparas para un examen real. ¿Qué serían en tu vida el «grupo de estudio», los «simulacros» y el «examen final»? ¿Alguna vez viste las respuestas antes de tiempo?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>